In [1]:
# --- BOOTSTRAP for "Save & Run All" (defines missing globals) ---
import os, glob, re, numpy as np, pandas as pd
from scipy import sparse

# 1) DATA_DIR + load data
if "DATA_DIR" not in globals():
    cand = [d for d in glob.glob("/kaggle/input/*")
            if any(k in d.lower() for k in ["polymer","neurips","open","opp"])]
    DATA_DIR = None
    for d in cand:
        if all(os.path.exists(os.path.join(d, f)) for f in ["train.csv","test.csv","sample_submission.csv"]):
            DATA_DIR = d; break
    if DATA_DIR is None:
        DATA_DIR = "/kaggle/input/neurips-open-polymer-prediction-2025"
    print("Using data dir:", DATA_DIR)

if "train" not in globals():
    train  = pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
    test   = pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
    sample = pd.read_csv(os.path.join(DATA_DIR,"sample_submission.csv"))

# 2) SMILES column
def _find_smiles(df):
    for c in df.columns:
        if c.lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("SMILES column not found")
if "SMI_COL" not in globals():
    SMI_COL = _find_smiles(train)

# 3) Targets present
TARGETS = ["Tg","FFV","Tc","Density","Rg"]
def _map_targets(df):
    m = {}
    for t in TARGETS:
        hits = [c for c in df.columns if c.lower()==t.lower()]
        if hits: m[t] = hits[0]
    return m
if "tmap" not in globals():
    tmap = _map_targets(train)
if "TARGETS_CAN" not in globals():
    TARGETS_CAN = [t for t in TARGETS if t in tmap]
print("Targets present:", TARGETS_CAN)

# 4) SMILES stats (if not already computed)
ATOM_TOKENS = ["Cl","Br","Si"]
SINGLE_CHARS = list("BCNOFPSI")
AROM_CHARS   = list("cnospb")
BOND_CHARS   = list("=#")
SYMS         = list("()[]")
EXTRA        = [".","%"]
RING_DIGITS  = list("123456789")

def _smiles_stats_vector(smiles: str):
    s = smiles
    L = len(s)
    counts = {}
    for tk in ATOM_TOKENS:
        counts[tk] = len(re.findall(tk, s))
        s = s.replace(tk, " ")
    for ch in SINGLE_CHARS: counts[ch] = s.count(ch)
    for ch in AROM_CHARS:   counts[f"ar_{ch}"] = s.count(ch)
    for ch in BOND_CHARS+SYMS+EXTRA: counts[ch] = s.count(ch)
    s_orig = smiles
    for d in RING_DIGITS: counts[f"ring_{d}"] = s_orig.count(d)

    ring_total = sum(counts[f"ring_{d}"] for d in RING_DIGITS)
    aromatic_total = sum(counts[f"ar_{ch}"] for ch in AROM_CHARS)
    caps_total = sum(counts.get(ch,0) for ch in SINGLE_CHARS) + sum(counts[tk] for tk in ATOM_TOKENS)
    bonds_total = sum(counts[ch] for ch in BOND_CHARS)
    parens_total = counts["("] + counts[")"]

    feats = [
        L, ring_total, aromatic_total, caps_total, bonds_total, parens_total,
        counts["["] + counts["]"], counts["."], counts["%"],
        counts.get("Cl",0), counts.get("Br",0), counts.get("Si",0),
        counts.get("C",0), counts.get("N",0), counts.get("O",0),
        counts.get("F",0), counts.get("P",0), counts.get("S",0), counts.get("I",0),
    ]
    def ratio(x): return (x / L) if L > 0 else 0.0
    feats += [ratio(ring_total), ratio(aromatic_total), ratio(caps_total),
              ratio(bonds_total), ratio(parens_total)]
    return np.array(feats, dtype=np.float32)

STAT_NAMES = [
    "len","rings","arom","caps","bonds","parens","brackets","dots","pct",
    "Cl","Br","Si","C","N","O","F","P","S","I",
    "r_rings","r_arom","r_caps","r_bonds","r_parens"
]

def _featurize_stats(smiles_list):
    M = len(smiles_list)
    X = np.zeros((M, len(STAT_NAMES)), dtype=np.float32)
    for i, smi in enumerate(smiles_list):
        X[i,:] = _smiles_stats_vector(str(smi))
    return X

if "train_smiles" not in globals():
    train_smiles = train[SMI_COL].astype(str).values
if "test_smiles" not in globals():
    test_smiles  = test[SMI_COL].astype(str).values
if "Xstats_train" not in globals():
    Xstats_train = _featurize_stats(train_smiles)
    Xstats_test  = _featurize_stats(test_smiles)


Using data dir: /kaggle/input/neurips-open-polymer-prediction-2025
Targets present: ['Tg', 'FFV', 'Tc', 'Density', 'Rg']


In [2]:
# List to confirm the wheel filename
!ls -lh /kaggle/input/rdkit-2025-3-3-cp311

# Install the wheel (Python 3.11 wheel shown here)
!pip install -q /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl

# Quick sanity check
import sys
print(sys.version)
from rdkit import Chem
from rdkit.Chem import Descriptors
print("RDKit OK:", Chem.MolFromSmiles("CCO") is not None)

total 34M
-rw-r--r-- 1 nobody nogroup 34M Jun 17 02:26 rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
RDKit OK: True


In [3]:
import os, gc, numpy as np, pandas as pd
from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import DataStructs

_morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

RANDOM_STATE = 42
N_SPLITS = 5

def compute_weights(y_df, cols):
    n = y_df[cols].notna().sum().astype(float)
    r = (y_df[cols].max() - y_df[cols].min()).astype(float).replace(0, 1.0)
    K = len(cols)
    inv_sqrt = 1/np.sqrt(n.clip(lower=1))
    norm = K * (inv_sqrt / inv_sqrt.sum())
    return (1.0/r) * norm

def approx_wmae(y_df, yhat_df, cols, weights):
    per_row = []
    for i in range(len(y_df)):
        s = 0.0
        for c in cols:
            yi = y_df.at[i,c]
            if pd.notna(yi):
                s += weights[c] * abs(yhat_df.at[i,c] - yi)
        per_row.append(s)
    denom = (y_df[cols].notna().sum(axis=1) > 0).sum()
    return float(np.sum(per_row) / max(denom,1))

y_full = train[[tmap[t] for t in TARGETS_CAN]].copy()
y_full.columns = TARGETS_CAN
weights = compute_weights(y_full, TARGETS_CAN)
print("Approx train-derived weights:", weights.to_dict())

HAS_RDKIT = False
try:
    from rdkit import Chem
    from rdkit.Chem import rdMolDescriptors, Descriptors
    HAS_RDKIT = True
    print("RDKit available.")
except Exception:
    print("RDKit not available; continuing without RDKit features.")

RD_DESC_NAMES = ["MolWt","TPSA","NumAromaticRings","NumAliphaticRings","RingCount",
                 "NumHAcceptors","NumHDonors","HeavyAtomCount","MolLogP"]

def rdkit_feats_for_smiles(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None, None
    fp = _morgan_gen.GetFingerprint(m)
    bits = np.zeros(2048, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, bits)
    desc = np.array([
        Descriptors.MolWt(m),
        Descriptors.TPSA(m),
        Descriptors.NumAromaticRings(m),
        Descriptors.NumAliphaticRings(m),
        Descriptors.RingCount(m),
        Descriptors.NumHAcceptors(m),
        Descriptors.NumHDonors(m),
        Descriptors.HeavyAtomCount(m),
        Descriptors.MolLogP(m),
    ], dtype=np.float32)
    return bits, desc

def build_rdkit_matrices(smiles_list):
    M = np.zeros((len(smiles_list), 2048), dtype=np.uint8)
    D = np.zeros((len(smiles_list), len(RD_DESC_NAMES)), dtype=np.float32)
    ok = np.ones(len(smiles_list), dtype=bool)
    for i, s in enumerate(smiles_list):
        bits, desc = rdkit_feats_for_smiles(str(s))
        if bits is None:
            ok[i] = False; continue
        M[i,:] = bits; D[i,:] = desc
    return sparse.csr_matrix(M), D, ok

if HAS_RDKIT:
    print("Building RDKit features...")
    M_train, D_train, _ = build_rdkit_matrices(train_smiles)
    M_test,  D_test,  _ = build_rdkit_matrices(test_smiles)
else:
    M_train = sparse.csr_matrix((len(train), 0))
    D_train = np.zeros((len(train), 0), dtype=np.float32)
    M_test  = sparse.csr_matrix((len(test), 0))
    D_test  = np.zeros((len(test), 0), dtype=np.float32)

extra_smiles = []
try:
    p = os.path.join(DATA_DIR, "train_supplement", "dataset2.csv")
    if os.path.exists(p):
        ds2 = pd.read_csv(p)
        smi_col2 = next(c for c in ds2.columns if c.lower() in ("smiles","smile","smiles_str","polymer_smiles"))
        extra_smiles = ds2[smi_col2].astype(str).tolist()
        print(f"Loaded {len(extra_smiles)} extra SMILES for TF-IDF augmentation.")
except Exception as e:
    print("dataset2.csv not used:", e)

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

VECT_SETTINGS = [
    dict(ngram_range=(2,5), min_df=2, max_features=200_000),
    dict(ngram_range=(2,6), min_df=1, max_features=300_000),
]
RIDGE_ALPHAS = [3.0, 7.0]
XGB_CFGS = [
    dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.85, colsample_bytree=0.6, reg_lambda=1.0),
    dict(n_estimators=900, learning_rate=0.03, max_depth=7, subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0),
]
if HAS_XGB and HAS_RDKIT:
    XGB_CFGS += [dict(n_estimators=1100, learning_rate=0.03, max_depth=8, subsample=0.85, colsample_bytree=0.7, reg_lambda=1.0)]

def fit_vectorizer(text_train_fold, vect_params, extra_smiles_list=None):
    tfidf = TfidfVectorizer(analyzer="char", **vect_params)
    if extra_smiles_list:
        fit_corpus = np.concatenate([text_train_fold, np.array(extra_smiles_list)])
        Xtr_full = tfidf.fit_transform(fit_corpus)
        return tfidf, Xtr_full[:len(text_train_fold)]
    else:
        return tfidf, tfidf.fit_transform(text_train_fold)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def try_config(target, vect_params, model_kind, model_params):
    tcol = tmap[target]
    mask = ~pd.isna(train[tcol])    
    y = train.loc[mask, tcol].values
    X_text = train_smiles[mask.values]           
    oof = np.zeros(mask.sum(), dtype=np.float32)
    pred_test = np.zeros(len(test), dtype=np.float32)

    idx_all = np.flatnonzero(mask.values)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_text), 1):
        X_tr_text, X_va_text = X_text[tr_idx], X_text[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        tfidf, Xtr_tfidf = fit_vectorizer(X_tr_text, vect_params, extra_smiles)
        Xva_tfidf = tfidf.transform(X_va_text)
        Xtst_tfidf = tfidf.transform(test_smiles)

        tr_rows = idx_all[tr_idx]
        va_rows = idx_all[va_idx]

        Xtr = sparse.hstack([
            Xtr_tfidf,
            sparse.csr_matrix(Xstats_train[tr_rows]),
            M_train[tr_rows],
            sparse.csr_matrix(D_train[tr_rows])
        ]).tocsr()
        Xva = sparse.hstack([
            Xva_tfidf,
            sparse.csr_matrix(Xstats_train[va_rows]),
            M_train[va_rows],
            sparse.csr_matrix(D_train[va_rows])
        ]).tocsr()
        Xtst = sparse.hstack([
            Xtst_tfidf,
            sparse.csr_matrix(Xstats_test),
            M_test,
            sparse.csr_matrix(D_test)
        ]).tocsr()

        if model_kind == "ridge":
            model = Ridge(alpha=model_params["alpha"], solver="lsqr", random_state=RANDOM_STATE)
            model.fit(Xtr, y_tr)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS
        else:
            if not HAS_XGB: raise RuntimeError("XGBoost not available")
            model = xgb.XGBRegressor(
                objective="reg:absoluteerror", eval_metric="mae",
                tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
                early_stopping_rounds=50, **model_params
            )
            model.fit(Xtr, y_tr, eval_set=[(Xva, y_va)], verbose=False)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS

        del tfidf, Xtr_tfidf, Xva_tfidf, Xtst_tfidf, Xtr, Xva, Xtst
        gc.collect()

    mae = mean_absolute_error(y, oof)
    w_contrib = float(weights[target] * mae)
    return oof, pred_test, w_contrib, mae

OOF_ours = pd.DataFrame(index=train.index, columns=TARGETS_CAN, dtype=float)
PRED_ours = pd.DataFrame({"id": test["id"].values})
for t in TARGETS_CAN: PRED_ours[t] = 0.0

for target in TARGETS_CAN:
    print(f"\n=== {target}: search configs ===")
    store = {}
    for vp in VECT_SETTINGS:
        for a in RIDGE_ALPHAS:
            name = f"ridge|ngr{vp['ngram_range']}_min{vp['min_df']}_a{a}"
            oof, pred, w, mae = try_config(target, vp, "ridge", {"alpha": a})
            store[name] = (oof, pred, w, mae)
            print(f"  {name:<42} w*MAE={w:.6f} MAE={mae:.6f}")
    for vp in VECT_SETTINGS:
        for j, xcfg in enumerate(XGB_CFGS):
            name = f"xgb|ngr{vp['ngram_range']}_min{vp['min_df']}_set{j+1}"
            oof, pred, w, mae = try_config(target, vp, "xgb", xcfg)
            store[name] = (oof, pred, w, mae)
            print(f"  {name:<42} w*MAE={w:.6f} MAE={mae:.6f}")

    best_name, (oof, pred, w, mae) = min(store.items(), key=lambda kv: kv[1][2])
    print(f"  -> selected: {best_name}  (w*MAE={w:.6f}, MAE={mae:.6f})")
    mask = ~pd.isna(train[tmap[target]])
    OOF_ours.loc[mask, target] = oof
    PRED_ours[target] = pred

our_overall = approx_wmae(y_full, OOF_ours[TARGETS_CAN], TARGETS_CAN, weights)
print("\n[OURS] Approx overall wMAE:", round(our_overall, 6))

PRED_final = PRED_ours.copy()
USE_PUBLIC_BLEND = False
try:
    from autogluon.tabular import TabularPredictor
    has_train_pkl = os.path.exists("/kaggle/input/neurips-preparedataset/train.pkl")
    has_test_pkl  = os.path.exists("/kaggle/working/test.pkl") or os.path.exists("/kaggle/input/neurips-preparedataset/test.pkl")
    has_any_model = any(os.path.exists(p) for p in [
        "/kaggle/input/neurips-train-autogluon-tg3/model/",
        "/kaggle/input/neurips-train-autogluon-ffv3/model/",
        "/kaggle/input/neurips-train-autogluon-tc3/model/",
        "/kaggle/input/neurips-train-autogluon-density3/model/",
        "/kaggle/input/neurips-train-autogluon-rg3/model/",
    ])
    if has_train_pkl and has_any_model:
        train_pub = pd.read_pickle("/kaggle/input/neurips-preparedataset/train.pkl")
        test_pub_path = "/kaggle/working/test.pkl" if os.path.exists("/kaggle/working/test.pkl") else "/kaggle/input/neurips-preparedataset/test.pkl"
        test_pub = pd.read_pickle(test_pub_path) if (test_pub_path and os.path.exists(test_pub_path)) else None

        MODELS = {
            "Tg":      "/kaggle/input/neurips-train-autogluon-tg3/model/",
            "Rg":      "/kaggle/input/neurips-train-autogluon-rg3/model/",
            "FFV":     "/kaggle/input/neurips-train-autogluon-ffv3/model/",
            "Density": "/kaggle/input/neurips-train-autogluon-density3/model/",
            "Tc":      "/kaggle/input/neurips-train-autogluon-tc3/model/",
        }

        OOF_public = pd.DataFrame(index=train.index, columns=TARGETS_CAN, dtype=float)
        PRED_public = pd.DataFrame({"id": test["id"].values})
        for t in TARGETS_CAN: PRED_public[t] = np.nan

        key = SMI_COL

        for t in TARGETS_CAN:
            if t in MODELS and os.path.exists(MODELS[t]):
                P = TabularPredictor.load(MODELS[t])
                tp = train_pub[[key]].copy()
                tp["_pred"] = P.predict(train_pub).values
                tp = tp.groupby(key, as_index=False)["_pred"].mean()
                OOF_public[t] = train[[key]].merge(tp, on=key, how="left")["_pred"].values

                if test_pub is not None and key in test_pub.columns:
                    tt = test_pub[[key]].copy()
                    tt["_pred"] = P.predict(test_pub).values
                    tt = tt.groupby(key, as_index=False)["_pred"].mean()
                    PRED_public[t] = test[[key]].merge(tt, on=key, how="left")["_pred"].values

        alphas = [0.7, 0.6, 0.5, 0.4]
        for t in TARGETS_CAN:
            tcol = tmap[t]; mask = ~pd.isna(train[tcol])
            y = train.loc[mask, tcol].values
            ours = OOF_ours.loc[mask, t].values.astype(float)
            pub  = OOF_public.loc[mask, t].values.astype(float)
            if np.isfinite(pub).sum() == len(pub):
                base = mean_absolute_error(y, ours)
                best_mae, best_alpha = base, 1.0
                for a in alphas:
                    m = mean_absolute_error(y, a*ours + (1-a)*pub)
                    if m < best_mae - 1e-6:
                        best_mae, best_alpha = m, a
                if best_alpha < 1.0 and np.isfinite(PRED_public[t]).any():
                    print(f"  {t}: blend ours:{best_alpha:.1f}/public:{1-best_alpha:.1f} (MAE {base:.5f}->{best_mae:.5f})")
                    PRED_final[t] = best_alpha*PRED_ours[t].values + (1-best_alpha)*np.nan_to_num(PRED_public[t].values, nan=PRED_ours[t].values)
                    USE_PUBLIC_BLEND = True
except Exception as e:
    print("Public blend skipped:", repr(e))
    PRED_final = PRED_ours.copy()
    USE_PUBLIC_BLEND = False

print("\nBlending used public models?", USE_PUBLIC_BLEND)

USE_DUPLICATE_FILL = True
if USE_DUPLICATE_FILL and (SMI_COL in train.columns) and (SMI_COL in test.columns):
    for t in TARGETS_CAN:
        tcol = tmap[t]
        vals = train[train[tcol].notnull()][[SMI_COL, tcol]].dropna().groupby(SMI_COL, as_index=False)[tcol].mean()
        m = test[SMI_COL].isin(vals[SMI_COL])
        if m.any():
            PRED_final.loc[m, t] = test.loc[m, [SMI_COL]].merge(vals, on=SMI_COL, how="left")[tcol].values

sub = pd.DataFrame({
    "id": test["id"].values,
    "Tg":      PRED_final["Tg"]      if "Tg"      in PRED_final else 0.0,
    "FFV":     PRED_final["FFV"]     if "FFV"     in PRED_final else 0.0,
    "Tc":      PRED_final["Tc"]      if "Tc"      in PRED_final else 0.0,
    "Density": PRED_final["Density"] if "Density" in PRED_final else 0.0,
    "Rg":      PRED_final["Rg"]      if "Rg"      in PRED_final else 0.0,
})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved /kaggle/working/submission.csv")

Approx train-derived weights: {'Tg': 0.00205237749659811, 'FFV': 0.6239248659724905, 'Tc': 2.2199754352943795, 'Density': 1.0640941761320573, 'Rg': 0.046558118429742154}
RDKit available.
Building RDKit features...
Loaded 7208 extra SMILES for TF-IDF augmentation.

=== Tg: search configs ===
  ridge|ngr(2, 5)_min2_a3.0                  w*MAE=0.117812 MAE=57.402806
  ridge|ngr(2, 5)_min2_a7.0                  w*MAE=0.112216 MAE=54.675977
  ridge|ngr(2, 6)_min1_a3.0                  w*MAE=0.117579 MAE=57.289061
  ridge|ngr(2, 6)_min1_a7.0                  w*MAE=0.112054 MAE=54.597296
  xgb|ngr(2, 5)_min2_set1                    w*MAE=0.107407 MAE=52.333035
  xgb|ngr(2, 5)_min2_set2                    w*MAE=0.109223 MAE=53.217971
  xgb|ngr(2, 5)_min2_set3                    w*MAE=0.108780 MAE=53.001723
  xgb|ngr(2, 6)_min1_set1                    w*MAE=0.108720 MAE=52.972529
  xgb|ngr(2, 6)_min1_set2                    w*MAE=0.108493 MAE=52.861958
  xgb|ngr(2, 6)_min1_set3                 